In [ ]:
from pathlib import Path

from imagematerials.__main__ import export_summary_netcdf, simulate_stocks
from imagematerials.vehicles.preprocessing import (
    preprocessing,
)
from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.concepts import vehicle_knowledge_graph

import prism


In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema.nc")

In [3]:
if not prep_fp.is_file():
    _, orig_prep_data = preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

In [4]:
import xarray as xr
import numpy as np
import pandas as pd
from imagematerials.util import dataset_to_array


# Copy dimensiomns from material_fractions for xr_maintenance_material
materials = prep_data['material_fractions'].coords["material"]
types = prep_data['material_fractions'].coords["Type"]

maintenance_material_pd : pd.DataFrame = pd.read_csv(
    "../data/raw/vehicles/standard_data/all_vehicle_maintenance_image.csv", index_col=0)

stacked_maintenance_material = maintenance_material_pd.set_index("Type").stack().rename_axis(index=["Type", "material"]).reset_index(name="value")

stacked_maintenance_material = stacked_maintenance_material.set_index(["Type", "material"])

stacked_maintenance_material_xr = stacked_maintenance_material.to_xarray()
maintenance_material = dataset_to_array(stacked_maintenance_material_xr, ["Type", "materials"], [])


In [ ]:
vehicle_knowledge_graph.rebroadcast_xarray_impute(maintenance_material, types.values)

<xarray.DataArray (Type: 53, materials: 10)> Size: 4kB
array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [1.06570556e-01, 8.70654668e-03, 1.47050000e-03, 1.18200000e-03,
        1.38000000e-04, 2.69986972e-02, 1.35951419e-02, 7.68615910e-02,
        1.25310600e-01, 0.00000000e+00],
       [1.06570556e-01, 8.70654668e-03, 1.47050000e-03, 1.18200000e-03,
        1.38000000e-04, 2.69986972e-02, 1.35951419e-02, 7.68615910e-02,
        1.25310600e-01, 0.00000000e+00],
       [1.06570556e-01, 8.70654668e-03, 1.47050000e-03, 1.18200000e-03,
        1.38000000e-04, 2.69986972e-02, 1.35951419e-02, 7.68615910e-02,
        1.25310600e-01, 0.00000000e+00],
       [1.06570556e-01, 8.70654668e-03, 1.47050000e-03, 1.18200000e-03,
        1.38000000e-04, 2.69986972e-02, 1.35951419e-02, 7.68615910e-02,
        1.25310600e-01, 0.00000000e+00],
       [1.06570556e-01, 8.70654668e-03, 1.47050000e-03, 1.18200000e-03,
        1.38000000e-04, 2.69986972e-02, 1.35951419e-02, 7.68615910e-02,
        1.25310600e-01, 0.00000000e+00],
       [1.06570556e-01, 8.70654668e-03, 1.47050000e-03, 1.18200000e-03,
        1.38000000e-04, 2.69986972e-02, 1.35951419e-02, 7.68615910e-02,
...
        0.00000000e+00, 0.00000000e+00, 7.44827586e-04, 0.00000000e+00,
        3.81663793e-02, 0.00000000e+00],
       [3.22448276e-03, 0.00000000e+00, 1.28844828e-03, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 7.44827586e-04, 0.00000000e+00,
        3.81663793e-02, 0.00000000e+00],
       [3.22448276e-03, 0.00000000e+00, 1.28844828e-03, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 7.44827586e-04, 0.00000000e+00,
        3.81663793e-02, 0.00000000e+00],
       [3.22448276e-03, 0.00000000e+00, 1.28844828e-03, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 7.44827586e-04, 0.00000000e+00,
        3.81663793e-02, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [1.21613459e-01, 0.00000000e+00, 2.08480215e-02, 1.12146247e-02,
        0.00000000e+00, 0.00000000e+00, 4.75797161e-02, 1.58813083e-02,
        2.05005545e-01, 8.09052282e-03],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00]])
Coordinates:
  * Type       (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'
  * materials  (materials) object 80B 'Aluminium' 'Co' 'Cu' ... 'Steel' 'Wood'

In [7]:

# Initialize xr_maintenance_material with zeros
xr_maintenance_material = xr.DataArray(
np.zeros((len(materials), len(types))),  # Shape based on dimensions
dims=("material", "Type"),
coords={"material": materials, "Type": types}
)

# Create a mapping from main vehicle types to their subtypes
vehicle_mapping = {
    "Cars": ["Cars", "Cars - BEV", "Cars - FCV", "Cars - HEV", "Cars - ICE", "Cars - PHEV", "Cars - Trolley"],
    "Trains": ["Trains", "High Speed Trains", "Freight Trains"],
    "Light Commercial Vehicles": ["Light Commercial Vehicles", "Light Commercial Vehicles - BEV", 
                                  "Light Commercial Vehicles - FCV", "Light Commercial Vehicles - HEV",
                                  "Light Commercial Vehicles - ICE", "Light Commercial Vehicles - PHEV", 
                                  "Light Commercial Vehicles - Trolley"],
    "Heavy Freight Trucks": ["Heavy Freight Trucks", "Heavy Freight Trucks - BEV", "Heavy Freight Trucks - FCV",
                             "Heavy Freight Trucks - HEV", "Heavy Freight Trucks - ICE", "Heavy Freight Trucks - PHEV",
                             "Heavy Freight Trucks - Trolley"],
    "Midi Buses": ["Midi Buses", "Midi Buses - BEV", "Midi Buses - FCV", "Midi Buses - HEV",
                   "Midi Buses - ICE", "Midi Buses - PHEV", "Midi Buses - Trolley"],
    "Regular Buses": ["Regular Buses", "Regular Buses - BEV", "Regular Buses - FCV", "Regular Buses - HEV",
                      "Regular Buses - ICE", "Regular Buses - PHEV", "Regular Buses - Trolley"],
}
# Check if the vehicle types in aggregated_df match the vehicle mapping
for main_type, subtypes in vehicle_mapping.items():
    print(f"Checking main type: {main_type}")
    print(f"Subtypes: {subtypes}")
    if main_type in maintenance_material.index:
        print(f"Aggregated data for {main_type}:")
        print(maintenance_material.loc[main_type, :])

# Now, try broadcasting the aggregated data
for main_type, subtypes in vehicle_mapping.items():
    if main_type in maintenance_material.index:
        for subtype in subtypes:
            if subtype in types:
                # Populate the correct values for the subtype
                xr_maintenance_material.loc[:, subtype] = maintenance_material.loc[main_type, :].values
                print(f"Assigned data for {main_type} to {subtype}")

# Check if the data is populated
print("Xarray after broadcasting:")
xr_maintenance_material

# Divide by lifetimes if data is populated
#if not xr_maintenance_material.isnull().all():
#    xr_maintenance_material /= prep_data["lifetimes"]
#else:
#    print("Xarray is still empty, check data alignment!")

# Check final Xarray
#print("Final Xarray after division by lifetimes:")
#print(xr_maintenance_material)

#xr_maintenance_material

Checking main type: Cars
Subtypes: ['Cars', 'Cars - BEV', 'Cars - FCV', 'Cars - HEV', 'Cars - ICE', 'Cars - PHEV', 'Cars - Trolley']


AttributeError: 'DataArray' object has no attribute 'index'

In [ ]:
prep_data["lifetimes"]["folded_norm"].coords["Type"]

<xarray.DataArray 'Type' (Type: 16)> Size: 2kB
array(['Bikes', 'Freight Planes', 'Freight Trains', 'Heavy Freight Trucks',
       'High Speed Trains', 'Inland Ships', 'Large Ships',
       'Light Commercial Vehicles', 'Medium Freight Trucks', 'Medium Ships',
       'Midi Buses', 'Passenger Planes', 'Regular Buses', 'Small Ships',
       'Trains', 'Very Large Ships'], dtype='<U25')
Coordinates:
  * Type     (Type) <U25 2kB 'Bikes' 'Freight Planes' ... 'Very Large Ships'

In [ ]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
main_model_normal = GenericMainModel(
    complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
    compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=True, 
    material=material)

In [ ]:
main_model_normal.simulate(simulation_timeline)

In [ ]:
main_model_factory = ModelFactory(
    prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).add(Maintenance
    ).finish()

In [ ]:
main_model_factory.simulate(simulation_timeline)